# Session 8 — Deployment & Launch
## Empire & Ink: AI Mughal Art Studio

**What you'll build:** A live, public URL for Empire & Ink. Commit to GitHub, deploy on Render — by the end, anyone in the world can visit your app.

---

## Setup

In [ ]:
!pip install gitpython requests

## Theory 1 — Git & Version Control

**Git** is a time machine for your code. Every `commit` is a snapshot you can return to.

| Command | What it does |
|---------|-------------|
| `git init` | Start tracking a folder |
| `git add .` | Stage all changes |
| `git commit -m "msg"` | Save a snapshot |
| `git push` | Upload to GitHub |
| `.gitignore` | Files Git should NEVER track |

**Critical:** Never commit `.env` files — they contain secret API keys that would be publicly visible on GitHub.

In [ ]:
git_workflow = [
    ("git init",                                  "Start a new repo in the current folder"),
    ("git add .",                                  "Stage all changed files"),
    ('git commit -m "Initial commit: Empire & Ink"', "Save a named snapshot"),
    ("git remote add origin https://github.com/you/empire-and-ink.git", "Link to GitHub"),
    ("git push -u origin main",                   "Upload code to GitHub"),
]

print("Git workflow — 5 steps to your first push:")
for i, (cmd, explanation) in enumerate(git_workflow, 1):
    print(f"  {i}. {cmd}")
    print(f"     → {explanation}")

## Theory 2 — What is Deployment?

**Deployment** means running your code on a server that is always on and connected to the internet.

**Render** is a cloud platform that:
1. Pulls your code from GitHub
2. Installs `requirements.txt`
3. Runs your start command: `uvicorn main:app --host 0.0.0.0 --port $PORT`
4. Gives you a public URL: `https://empire-and-ink.onrender.com`

Environment variables in production: instead of a `.env` file, you set them in the Render dashboard. Your code reads them the same way with `os.getenv()`.

In [ ]:
render_yaml = (
    "services:\n"
    "  - type: web\n"
    "    name: empire-and-ink\n"
    "    env: python\n"
    "    buildCommand: pip install -r requirements.txt\n"
    "    startCommand: uvicorn main:app --host 0.0.0.0 --port $PORT\n"
    "    envVars:\n"
    "      - key: GEMINI_API_KEY\n"
    "        sync: false\n"
    "      - key: SUPABASE_URL\n"
    "        sync: false\n"
    "      - key: SUPABASE_KEY\n"
    "        sync: false\n"
    "      - key: SUPABASE_SERVICE_KEY\n"
    "        sync: false\n"
)
print(render_yaml)

## Theory 3 — Environment Variables in Production

In development: secrets live in `.env` (never committed).
In production: secrets live in the hosting platform's dashboard.

```python
# config.py — same code works in both environments
import os
from dotenv import load_dotenv

load_dotenv()  # loads .env if it exists (dev); no-op in production

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
SUPABASE_URL   = os.getenv("SUPABASE_URL")
```

**12-Factor App principle:** store configuration in the environment, never in the code.

In [ ]:
import os

required_vars = ["GEMINI_API_KEY", "SUPABASE_URL", "SUPABASE_KEY", "SUPABASE_SERVICE_KEY"]
print("Environment variable status check:")
for var in required_vars:
    status = "SET" if os.getenv(var) else "MISSING"
    print(f"  {status:8} {var}")

---
## In-Class Exercises

### Exercise 1 — Explain render.yaml line by line

In [ ]:
render_yaml_explained = {
    "type: web":       "A web service = an always-on HTTP server with a public URL",
    "name: empire-and-ink": "The service name on Render — appears in your URL",
    "env: python":     "Render uses its Python build environment (installs Python for you)",
    "buildCommand":    "Runs once at deploy time to install all Python packages",
    "startCommand":    "Runs to start the server; $PORT is assigned by Render automatically",
    "envVars":         "Secret variables set in the Render dashboard, not stored in this file",
    "sync: false":     "Never auto-sync from YAML — always set manually in dashboard for security",
}

print("render.yaml — every line explained:")
for key, explanation in render_yaml_explained.items():
    print(f"  {key}:")
    print(f"    {explanation}")

### Exercise 2 — Simulate git workflow with GitPython

In [ ]:
import os, tempfile
import git

with tempfile.TemporaryDirectory() as tmpdir:
    repo = git.Repo.init(tmpdir)
    print(f"1. git init: Initialised repo at {tmpdir}")

    readme = os.path.join(tmpdir, "README.md")
    with open(readme, "w") as f:
        f.write("# Empire & Ink\nAI Mughal Art Studio\n")
    print("2. Created README.md")

    # YOUR CODE HERE — stage and commit the file
    # repo.index.add(["README.md"])
    # commit = repo.index.commit("Initial commit: Empire & Ink")
    # print(f"3. Committed: {commit.hexsha[:8]} — {commit.message}")
    repo.index.add(["README.md"])
    commit = repo.index.commit("Initial commit: Empire & Ink")
    print(f"3. Committed: {commit.hexsha[:8]} — {commit.message}")

### Exercise 3 — Pre-deploy environment variable checker

In [ ]:
import os

def check_deploy_env():
    required = ["GEMINI_API_KEY", "SUPABASE_URL", "SUPABASE_KEY", "SUPABASE_SERVICE_KEY"]
    results = {var: bool(os.getenv(var)) for var in required}
    all_ok  = all(results.values())

    print("Pre-deploy environment check:")
    print("=" * 42)
    for var, is_set in results.items():
        print(f"  {'OK' if is_set else 'MISSING':8} {var}")
    print("=" * 42)
    if all_ok:
        print("All variables set — ready to deploy!")
    else:
        missing = [v for v, ok in results.items() if not ok]
        print(f"ERROR: {len(missing)} variable(s) missing. Set them before deploying.")
    return results

check_deploy_env()

### Exercise 4 — Review requirements.txt and explain each package

In [ ]:
requirements_explained = {
    "fastapi":            "The web framework — HTTP routing and request parsing",
    "uvicorn[standard]":  "ASGI server that runs FastAPI and listens for connections",
    "python-multipart":   "Required for file uploads (UploadFile) in FastAPI",
    "jinja2":             "Template engine for rendering HTML pages server-side",
    "google-genai":       "Google SDK for Gemini (text) and Imagen (image) AI models",
    "supabase":           "Official Supabase SDK for database, storage and auth",
    "python-dotenv":      "Loads .env files into os.getenv() in development",
    "pydantic[email]":    "Data validation — also validates email format in auth schemas",
    "pillow":             "Image processing: open, save, resize PNG/JPEG files",
}

print("requirements.txt — each package explained:")
for pkg, explanation in requirements_explained.items():
    print(f"  {pkg}")
    print(f"    -> {explanation}")

---
## Post-Class Exercises

### Challenge 1 — health_check() pinging the live URL

In [ ]:
import requests, time

def health_check(url: str, retries: int = 5, delay: int = 10) -> bool:
    endpoint = url.rstrip("/") + "/api/health"
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(endpoint, timeout=10)
            if r.status_code == 200:
                print(f"PASS on attempt {attempt}: {r.json()}")
                return True
            print(f"Attempt {attempt}: status {r.status_code}")
        except requests.RequestException as e:
            print(f"Attempt {attempt}: {e}")
        if attempt < retries:
            time.sleep(delay)
    print("Health check FAILED after all retries.")
    return False

# Replace with your real Render URL after deployment
health_check("https://empire-and-ink.onrender.com", retries=1, delay=0)

### Challenge 2 — Document 3 v2 feature proposals

In [ ]:
def get_v2_features():
    return [
        {
            "name": "Social gallery",
            "description": "A public gallery where users can share and like each other's Mughal art",
            "technical_approach": "Add is_public column to galleries. New GET /api/explore endpoint. "
                                  "POST /api/gallery/{id}/like increments a likes counter.",
            "effort": 3,
        },
        # YOUR CODE HERE — add 2 more v2 feature proposals
        # Each needs: name, description, technical_approach, effort (1-5)
        {
            "name": "YOUR FEATURE 2",
            "description": "Describe it here",
            "technical_approach": "How you would build it technically",
            "effort": 2,
        },
        {
            "name": "YOUR FEATURE 3",
            "description": "Describe it here",
            "technical_approach": "How you would build it technically",
            "effort": 4,
        },
    ]

for f in get_v2_features():
    print(f"[Effort {f['effort']}/5] {f['name']}: {f['description'][:60]}...")

### Challenge 3 — Write git commit messages for all 8 sessions

In [ ]:
# Write a meaningful git commit message for each session's deliverable.
# Convention: <type>(<scope>): <description>
# Types: feat, fix, chore, docs, refactor

commit_messages = [
    "chore(setup): initialise project with Python venv and describe_artwork.py",
    "feat(api): add FastAPI with health, gallery, and generate stub endpoints",
    "feat(db): connect Supabase and implement gallery CRUD service",
    "feat(ai): add Gemini prompt enhancer and Imagen image generator",
    "feat(pipeline): implement full generate and transform orchestrator pipelines",
    "feat(auth): add Supabase JWT auth with Depends() protected routes",
    "feat(frontend): build complete Jinja2 + Tailwind + JS frontend with gallery lightbox",
    "chore(deploy): add render.yaml, .gitignore, and production configuration",
]

for i, msg in enumerate(commit_messages, 1):
    print(f"Session {i}: {msg}")

### Challenge 4 — Describe the full architecture in a structured format

In [ ]:
def describe_architecture():
    layers = {
        "Frontend":       "Jinja2 HTML templates served by FastAPI. Tailwind CSS utilities. "
                          "Vanilla JS with Auth object + apiFetch() wrapper. localStorage for JWT tokens.",
        "Backend API":    "FastAPI with Pydantic validation. Routes in app/api/routes.py. "
                          "All gallery/generate routes protected by get_current_user() Depends().",
        "AI Pipeline":    "google-genai SDK: Gemini Flash (text) enhances prompts, Imagen 3 generates art, "
                          "Gemini Flash multimodal transforms uploaded photos.",
        "Database":       "Supabase PostgreSQL. galleries table stores metadata. "
                          "Row Level Security ensures users only see their own rows.",
        "File Storage":   "Supabase Storage bucket 'gallery-images'. Images uploaded as bytes, "
                          "served via public CDN URLs.",
        "Authentication": "Supabase Auth email/password. JWT tokens issued on login. "
                          "FastAPI HTTPBearer validates tokens on every protected request.",
        "Deployment":     "GitHub repository + Render web service. Secrets in Render dashboard. "
                          "Auto-deploys on every push to the main branch.",
    }
    print("=" * 60)
    print("EMPIRE & INK — FULL ARCHITECTURE")
    print("=" * 60)
    for layer, description in layers.items():
        print(f"\n{layer.upper()}:")
        words = description.split()
        line = "  "
        for word in words:
            if len(line) + len(word) > 68:
                print(line)
                line = "  " + word + " "
            else:
                line += word + " "
        if line.strip(): print(line)
    print("=" * 60)

describe_architecture()

---
## Congratulations — Empire & Ink is live!

Over 8 sessions you went from zero Python to a live AI-powered Mughal art generator with:
- A FastAPI backend with JWT authentication
- Google Gemini AI for prompt enhancement and Imagen 3 for art generation
- Supabase for user data, image storage, and Row Level Security
- A full HTML/CSS/JS frontend with gallery, lightbox, and download

**Share your live URL, collect feedback, and start building v2 features!**